# Monitoreo del modelo Adult Income

Este notebook ejecuta el flujo completo de monitoreo en orden con Python 3.13.14. Ejecute las celdas de arriba hacia abajo.

> Antes de abrir Jupyter, active el entorno virtual desde PowerShell con `./.venv/Scripts/Activate.ps1` y abra el notebook usando ese entorno.

## 1. Verificar el entorno y la carpeta del proyecto

La ruta mostrada debe terminar en `respositorio25-08-2026` y el ejecutable debe pertenecer a `.venv`.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
print('Carpeta:', PROJECT_DIR)
print('Python:', sys.executable)
print('Versión:', sys.version.split()[0])

required = [
    PROJECT_DIR / 'requirements-monitoring.txt',
    PROJECT_DIR / 'src' / 'monitoring.py',
    PROJECT_DIR / 'src' / 'simulate_production.py',
    PROJECT_DIR / 'resultado_pipeline' / 'adult_clean.csv',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Faltan archivos: {missing}'
assert '.venv' in sys.executable, 'Seleccione como kernel el Python de .venv.'
print('Entorno y archivos verificados correctamente.')

Carpeta: c:\Users\c3283\Desktop\INCOEX\respositorio25-08-2026
Python: c:\Users\c3283\Desktop\INCOEX\fase6 proyecto integrador\.venv\Scripts\python.exe
Versión: 3.13.14
Entorno y archivos verificados correctamente.


## 2. Instalar dependencias

`%pip` instala las dependencias en el mismo entorno utilizado por el notebook.

In [2]:
%pip install -r requirements-monitoring.txt

  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached fastapi-0.139.0-py3-none-any.whl.metadata (26 kB)
  Using cached uvicorn-0.29.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached prometheus_client-0.26.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached starlette-1.6.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached pydantic_core-2.46.5-cp313-cp313-win_amd64.whl.metadata (6.7 kB)
Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl (11.0 MB)
Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl (12.3 MB)
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.6 MB ? eta -:--:--
   - -----------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Ejecutar las pruebas de monitoreo

El resultado esperado es `12 passed`.

In [3]:
!{sys.executable} -m pytest tests/test_monitoring.py -v -p no:cacheprovider

"c:\Users\c3283\Desktop\INCOEX\fase6" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


## 4. Ejecutar todas las pruebas del proyecto (recomendado)

Esta comprobación es más amplia. El resultado esperado actualmente es `77 passed`.

In [4]:
!{sys.executable} -m pytest tests -v -p no:cacheprovider

"c:\Users\c3283\Desktop\INCOEX\fase6" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


## 5. Generar seis lotes simulados

Se generan 6 lotes de 1.000 filas con semilla 42. El drift aumenta progresivamente entre los lotes.

In [5]:
!{sys.executable} -m src.simulate_production --batches 6 --batch-size 1000 --random-state 42

"c:\Users\c3283\Desktop\INCOEX\fase6" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


## 6. Revisar el resumen de los lotes

La tabla permite observar desde qué lote aparece drift y cómo cambian las métricas del modelo.

In [6]:
import pandas as pd

summary_path = PROJECT_DIR / 'resultado_pipeline' / 'monitoring' / 'monitoring_summary.csv'
summary = pd.read_csv(summary_path)
display(summary)

,batch_id,drift_strength,drift_detected,high_drift_features,precision,recall,f1,roc_auc
0,1,0.0,False,NaN,0.744444,0.848101,0.792899,0.960430
1,2,0.2,False,NaN,0.560117,0.760956,0.645270,0.850068
2,3,0.4,True,"capital-gain,hours-per-week",0.477833,0.772908,0.590563,0.796754
3,4,0.6,True,"capital-gain,hours-per-week",0.407563,0.757812,0.530055,0.734443
4,5,0.8,True,"age,capital-gain,hours-per-week",0.302326,0.793427,0.437824,0.695188
5,6,1.0,True,"age,capital-gain,hours-per-week",0.287902,0.764228,0.418242,0.642228


## 7. Generar el reporte general

El reporte compara el dataset de referencia con los 6.000 registros simulados y guarda el resultado en JSON.

In [7]:
!{sys.executable} -m src.monitoring --reference resultado_pipeline/adult_clean.csv --production resultado_pipeline/monitoring/production_batch.csv --output resultado_pipeline/monitoring/monitoring_report.json

"c:\Users\c3283\Desktop\INCOEX\fase6" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


## 8. Interpretar el reporte

Se muestran las variables con drift alto y las métricas generales del modelo.

In [8]:
import json

report_path = PROJECT_DIR / 'resultado_pipeline' / 'monitoring' / 'monitoring_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))

print('Drift detectado:', report['data_monitoring']['drift_detected'])
print('Variables con drift alto:', report['data_monitoring']['high_drift_features'])
print('Métricas del modelo:')
display(pd.Series(report['model_monitoring']))

Drift detectado: True
Variables con drift alto: ['capital-gain', 'hours-per-week']
Métricas del modelo:


status                      evaluated
rows                             6000
valid_predictions                6000
positive_prediction_rate     0.450833
average_probability          0.499219
probability_std              0.424364
ground_truth_available           True
evaluated_rows                   6000
precision                    0.420333
recall                       0.781981
f1                           0.546766
roc_auc                      0.760065
dtype: object

## 9. Iniciar la API en el puerto 8001

La siguiente celda inicia Uvicorn en segundo plano, sin `--reload`, para evitar procesos adicionales dentro del notebook. Si el puerto ya está ocupado, primero detenga la otra API.

In [9]:
import subprocess
import time
import urllib.request

if 'api_process' in globals() and api_process.poll() is None:
    print('La API iniciada por este notebook ya está activa.')
else:
    api_process = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'src.api.main:app', '--host', '127.0.0.1', '--port', '8001'],
        cwd=PROJECT_DIR,
    )
    for _ in range(20):
        if api_process.poll() is not None:
            raise RuntimeError('La API no pudo iniciar. Revise si el puerto 8001 está ocupado.')
        try:
            with urllib.request.urlopen('http://127.0.0.1:8001/health', timeout=1) as response:
                if response.status == 200:
                    break
        except Exception:
            time.sleep(0.5)
    else:
        raise RuntimeError('La API no respondió dentro del tiempo esperado.')
    print('API activa en http://127.0.0.1:8001')

API activa en http://127.0.0.1:8001


## 10. Consultar salud y métricas del sistema

Rutas disponibles:

- [Estado del modelo](http://127.0.0.1:8001/health)
- [Métricas del sistema](http://127.0.0.1:8001/monitoring/system)
- [Swagger](http://127.0.0.1:8001/docs)
- [Especificación OpenAPI](http://127.0.0.1:8001/openapi.json)

In [10]:
def get_json(url):
    with urllib.request.urlopen(url, timeout=5) as response:
        return json.loads(response.read().decode('utf-8'))

print('Estado del modelo:')
display(get_json('http://127.0.0.1:8001/health'))
print('Métricas del sistema:')
display(get_json('http://127.0.0.1:8001/monitoring/system'))

Estado del modelo:


{'status': 'ok',
 'algorithm': 'hist_gradient_boosting',
 'threshold': 0.645,
 'model_version': '225a40e0'}

Métricas del sistema:


{'total_requests': 2,
 'successful_requests': 2,
 'error_requests': 0,
 'uptime_seconds': 5.489,
 'availability': 1.0,
 'error_rate': 0.0,
 'throughput_requests_per_second': 0.364364,
 'average_latency_ms': 4.5257,
 'p95_latency_ms': 7.5025}

## 11. Detener la API iniciada por el notebook

Ejecute esta celda cuando termine. Solo detiene el proceso creado en la sección 9.

In [11]:
if 'api_process' in globals() and api_process.poll() is None:
    api_process.terminate()
    api_process.wait(timeout=10)
    print('API detenida.')
else:
    print('No hay una API iniciada por este notebook.')

API detenida.


## Equivalente desde PowerShell

Si desea iniciar la API fuera del notebook, use una terminal independiente:

```powershell
./.venv/Scripts/Activate.ps1
python -m uvicorn src.api.main:app --reload --host 127.0.0.1 --port 8001
```

En otra terminal puede consultar las métricas:

```powershell
Invoke-RestMethod -Uri "http://127.0.0.1:8001/monitoring/system" -Method Get
```